In [1]:
!pip install -q langchain langchain-community langchain-groq chromadb sentence-transformers pypdf

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Attention.pdf to Attention (1).pdf


In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_file = "Attention.pdf"

loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("Pages loaded:", len(documents))

/tmp/ipykernel_2765/653161642.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages loaded: 15


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

print("Chunks created:", len(docs))

Chunks created: 103


In [5]:
!pip install -q langchain-text-splitters

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_2765/2985419238.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    docs,
    embedding
)

print("Vector database created successfully!")

Vector database created successfully!


In [14]:
def answer_from_pdf(query):

    docs = db.as_retriever().invoke(query)

    context = "\n".join(doc.page_content for doc in docs)

    prompt = f"""
Use ONLY the context below to answer the question.

Context:
{context}

Question:
{query}

If the answer is not in the context, reply exactly:

I don't know
"""

    return llm.invoke(prompt).content

In [15]:
def adaptive_answer(question, max_tries=3):

    query = question

    for attempt in range(1, max_tries + 1):

        print(f"Attempt {attempt} with query: {query}")

        answer = answer_from_pdf(query)

        if "i don't know" not in answer.lower():

            return answer

        query = llm.invoke(
            f"Rephrase this search query differently: {query}"
        ).content

    return "Could not find an answer after several tries."

In [16]:
question = "Why is self-attention important?"

print(adaptive_answer(question))

Attempt 1 with query: Why is self-attention important?
Self-attention is important because it allows the model to yield more interpretable models. It also enables the model to follow long-distance dependencies, as shown in Figure 3, where the attention mechanism attends to a distant dependency of the verb 'making'.
